In [ ]:
try:
  import Bio
  import pandas as pd
except:
  !pip install pandas
  !pip install biopython
  import Bio
  import pandas as pd
import requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 12.5 MB/s eta 0:00:00


In [ ]:
import requests
import time

def getGoTermsFromUniprot(uniprot_ids: list) -> dict:
    """
    Retrieves from UniProt database the GO terms associated
    with each protein in the list, with a 20-second delay between each request.

    Parameters:
    uniprot_ids (list): A list of UniProt IDs of the proteins.

    Returns:
    dict: A dictionary where keys are UniProt IDs and values are lists of GO terms.
    """
    base_url = "https://rest.uniprot.org/uniprotkb/"
    go_terms_dict = {}
    total_ids = len(uniprot_ids)
    start_time = time.time()

    for index, uniprot_id in enumerate(uniprot_ids, start=1):
        url = f"{base_url}{uniprot_id}.json"
        response = requests.get(url)

        if response.status_code != 200:
            print(f"Error fetching data for {uniprot_id}: {response.status_code}")
            go_terms_dict[uniprot_id] = []
        else:
            data = response.json()
            go_terms = []

            # Extract GO terms from the JSON response
            for dbReference in data.get('uniProtKBCrossReferences', []):
                if dbReference.get('database') == 'GO':
                    go_terms.append(dbReference.get('id'))

            go_terms_dict[uniprot_id] = go_terms

        # Calculate elapsed time and remaining IDs
        elapsed_time = time.time() - start_time
        remaining_ids = total_ids - index
        if remaining_ids % 50 == 0:
            print(f"Traités en {elapsed_time:.2f} sec : {index}, restants : {remaining_ids}")

        go_terms_dict_res = {}
        for item in go_terms_dict:
            go_terms_dict_res[item] = ";".join(go_terms_dict[item])

    return go_terms_dict_res

# Exemple d'utilisation
uniprot_ids = ["P12345", "Q6GZX4", "P20382", "Q14833"]  # Ajoutez vos identifiants ici
go_terms_dict = getGoTermsFromUniprot(uniprot_ids)
print(go_terms_dict)


Traités en 1.95 sec : 4, restants : 0
{'P12345': 'GO:0005759;GO:0005739;GO:0005886;GO:0016212;GO:0004069;GO:0030170;GO:0006103;GO:0006533;GO:0006531;GO:0006536;GO:0006869;GO:0006457', 'Q6GZX4': 'GO:0046782', 'P20382': 'GO:0005576;GO:0005634;GO:0045202;GO:0030354;GO:0031777;GO:0030154;GO:0007268;GO:0007631;GO:0032227;GO:0007218;GO:0007283', 'Q14833': 'GO:0031410;GO:0005886;GO:0098793;GO:0001640;GO:0004930;GO:0008066;GO:0007196;GO:0007268;GO:0007216;GO:0007269;GO:0043410;GO:0043523;GO:0051966'}


In [ ]:
df_brain = pd.read_csv('/content/tissue_category_rna_brain_Tissue.tsv', sep='\t')
df_brain_reduced = df_brain[["Uniprot", "Gene description"]]
df_brain_reduced["Organ"] = ["Brain"] * len(df_brain_reduced)
df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]
df_brain_reduced = df_brain_reduced.dropna()

<ipython-input-70-8675caad30dc>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Organ"] = ["Brain"] * len(df_brain_reduced)
<ipython-input-70-8675caad30dc>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]


In [ ]:
brain_go_dict = getGoTermsFromUniprot(df_brain_reduced["Uniprot"].tolist())
df_brain_reduced["GO"] = df_brain_reduced["Uniprot"].map(brain_go_dict)
df_brain_reduced

Traités en 1.48 sec : 3, restants : 2150
Traités en 26.62 sec : 53, restants : 2100
Traités en 50.95 sec : 103, restants : 2050
Traités en 75.56 sec : 153, restants : 2000
Traités en 100.11 sec : 203, restants : 1950
Traités en 124.54 sec : 253, restants : 1900
Traités en 149.36 sec : 303, restants : 1850
Traités en 174.08 sec : 353, restants : 1800
Traités en 198.97 sec : 403, restants : 1750
Traités en 223.77 sec : 453, restants : 1700
Traités en 248.69 sec : 503, restants : 1650
Traités en 273.43 sec : 553, restants : 1600
Traités en 298.38 sec : 603, restants : 1550
Traités en 323.68 sec : 653, restants : 1500
Traités en 348.28 sec : 703, restants : 1450
Traités en 373.28 sec : 753, restants : 1400
Traités en 398.13 sec : 803, restants : 1350
Traités en 422.84 sec : 853, restants : 1300
Traités en 447.48 sec : 903, restants : 1250
Traités en 472.19 sec : 953, restants : 1200
Traités en 497.31 sec : 1003, restants : 1150
Traités en 521.94 sec : 1053, restants : 1100
Traités en 546.8

,Uniprot,Gene description,Organ,GO
0,O43612,Hypocretin neuropeptide precursor,Brain,GO:0005576;GO:0099013;GO:0048471;GO:0098794;GO...
1,P01185,Arginine vasopressin,Brain,GO:0030669;GO:0005829;GO:0030425;GO:0005576;GO...
2,P20382,Pro-melanin concentrating hormone,Brain,GO:0005576;GO:0005634;GO:0045202;GO:0030354;GO...
3,Q14833,Glutamate metabotropic receptor 4,Brain,GO:0031410;GO:0005886;GO:0098793;GO:0001640;GO...
4,Q9BZE3,BarH like homeobox 1,Brain,GO:0000785;GO:0005634;GO:0001228;GO:0000981;GO...
...,...,...,...,...
2192,Q96CK0,Zinc finger protein 653,Brain,GO:0005576;GO:0005634;GO:0050682;GO:0003677;GO...
2193,Q7Z570,Zinc finger protein 804A,Brain,GO:0005737;GO:1901588;GO:0043198;GO:0043197;GO...
2194,O75541,Zinc finger protein 821,Brain,GO:0005634;GO:0003700;GO:0046872;GO:0000978;GO...
2195,Q8NBB4,Zinc finger and SCAN domain containing 1,Brain,GO:0000785;GO:0005634;GO:0000981;GO:0046872;GO...


In [ ]:
df_brain_reduced.to_csv("brain_go.csv", index=False)

In [ ]:
df_brain = pd.read_csv('/content/tissue_category_rna_heart.tsv', sep='\t')
df_brain_reduced = df_brain[["Uniprot", "Gene description"]]
df_brain_reduced["Organ"] = ["Heart"] * len(df_brain_reduced)
df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]
df_brain_reduced = df_brain_reduced.dropna()
brain_go_dict = getGoTermsFromUniprot(df_brain_reduced["Uniprot"].tolist())
df_brain_reduced["GO"] = df_brain_reduced["Uniprot"].map(brain_go_dict)
df_brain_reduced.to_csv("heart_go.csv", index=False)
df_brain_reduced

<ipython-input-76-b204c96297cf>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Organ"] = ["Heart"] * len(df_brain_reduced)
<ipython-input-76-b204c96297cf>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]


Traités en 7.27 sec : 14, restants : 400
Traités en 32.53 sec : 64, restants : 350
Traités en 59.09 sec : 114, restants : 300
Traités en 85.61 sec : 164, restants : 250
Traités en 112.23 sec : 214, restants : 200
Traités en 139.27 sec : 264, restants : 150
Traités en 166.04 sec : 314, restants : 100
Traités en 193.20 sec : 364, restants : 50
Traités en 219.92 sec : 414, restants : 0


,Uniprot,Gene description,Organ,GO
0,Q01449,Myosin light chain 7,Heart,GO:0031672;GO:0005737;GO:0005829;GO:0043197;GO...
1,P01160,Natriuretic peptide A,Heart,GO:0042995;GO:0062023;GO:0005737;GO:0005576;GO...
2,P19429,"Troponin I3, cardiac type",Heart,GO:0097512;GO:1990584;GO:0005829;GO:0030017;GO...
3,Q14896,Myosin binding protein C3,Heart,GO:0031672;GO:0014705;GO:0097512;GO:0005829;GO...
4,P16860,Natriuretic peptide B,Heart,GO:0005737;GO:0005576;GO:0005615;GO:0032991;GO...
...,...,...,...,...
414,Q9NY65,Tubulin alpha 8,Heart,GO:0001669;GO:0005737;GO:0005874;GO:0015630;GO...
415,O14957,"Ubiquinol-cytochrome c reductase, complex III ...",Heart,GO:0005743;GO:0005739;GO:0045275;GO:0009055;GO...
416,A0A096LP55,Ubiquinol-cytochrome c reductase hinge protein...,Heart,GO:0005743;GO:0005739;GO:0045275;GO:0006122
417,O14949,Ubiquinol-cytochrome c reductase complex III s...,Heart,GO:0005743;GO:0005739;GO:0045275;GO:0045333;GO...


In [ ]:
df_brain = pd.read_csv('/content/tissue_category_rna_kidney_Tissue.tsv', sep='\t')
df_brain_reduced = df_brain[["Uniprot", "Gene description"]]
df_brain_reduced["Organ"] = ["Kidney"] * len(df_brain_reduced)
df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]
df_brain_reduced = df_brain_reduced.dropna()
brain_go_dict = getGoTermsFromUniprot(df_brain_reduced["Uniprot"].tolist())
df_brain_reduced["GO"] = df_brain_reduced["Uniprot"].map(brain_go_dict)
df_brain_reduced.to_csv("kidney_go.csv", index=False)
df_brain_reduced

<ipython-input-77-45607bfde49e>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Organ"] = ["Kidney"] * len(df_brain_reduced)
<ipython-input-77-45607bfde49e>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]


Traités en 24.56 sec : 46, restants : 400
Traités en 51.57 sec : 96, restants : 350
Traités en 77.87 sec : 146, restants : 300
Traités en 104.27 sec : 196, restants : 250
Traités en 130.47 sec : 246, restants : 200
Traités en 157.10 sec : 296, restants : 150
Traités en 182.94 sec : 346, restants : 100
Traités en 209.23 sec : 396, restants : 50
Traités en 235.48 sec : 446, restants : 0


,Uniprot,Gene description,Organ,GO
0,Q08AH3,Acyl-CoA synthetase medium chain family member 2A,Kidney,GO:0005759;GO:0005739;GO:0005524;GO:0018858;GO...
1,Q13621,Solute carrier family 12 member 1,Kidney,GO:0016324;GO:0070062;GO:0016020;GO:0005886;GO...
2,Q8WUU8,Transmembrane protein 174,Kidney,GO:0016324;GO:0005789;GO:0055062
3,P07911,Uromodulin,Kidney,GO:0016324;GO:0016323;GO:0009986;GO:0060170;GO...
4,O60656,UDP glucuronosyltransferase family 1 member A9,Kidney,GO:0005783;GO:0005789;GO:0019899;GO:0004857;GO...
...,...,...,...,...
455,Q14D04,Ventricular zone expressed PH domain containing 1,Kidney,GO:0005886;GO:0010314;GO:0060392;GO:0030512;GO...
456,Q3MJ13,WD repeat domain 72,Kidney,GO:0005737;GO:0005768;GO:0005634;GO:0036305;GO...
457,Q96NZ8,"WAP, follistatin/kazal, immunoglobulin, kunitz...",Kidney,GO:0005615;GO:0048019;GO:0004867;GO:0050431;GO...
458,Q96J92,WNK lysine deficient protein kinase 4,Kidney,GO:0005923;GO:0044297;GO:0005737;GO:0005829;GO...


In [ ]:
df_brain = pd.read_csv('/content/tissue_category_rna_liver_Tissue.tsv', sep='\t')
df_brain_reduced = df_brain[["Uniprot", "Gene description"]]
df_brain_reduced["Organ"] = ["Liver"] * len(df_brain_reduced)
df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]
df_brain_reduced = df_brain_reduced.dropna()
brain_go_dict = getGoTermsFromUniprot(df_brain_reduced["Uniprot"].tolist())
df_brain_reduced["GO"] = df_brain_reduced["Uniprot"].map(brain_go_dict)
df_brain_reduced.to_csv("Liver_go.csv", index=False)
df_brain_reduced

<ipython-input-78-1b7cb2f319c1>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Organ"] = ["Liver"] * len(df_brain_reduced)
<ipython-input-78-1b7cb2f319c1>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]


Traités en 1.62 sec : 3, restants : 950
Traités en 29.95 sec : 53, restants : 900
Traités en 57.48 sec : 103, restants : 850
Traités en 84.76 sec : 153, restants : 800
Traités en 111.19 sec : 203, restants : 750
Traités en 138.15 sec : 253, restants : 700
Traités en 165.23 sec : 303, restants : 650
Traités en 191.88 sec : 353, restants : 600
Traités en 219.16 sec : 403, restants : 550
Traités en 245.99 sec : 453, restants : 500
Traités en 273.36 sec : 503, restants : 450
Traités en 299.49 sec : 553, restants : 400
Traités en 326.43 sec : 603, restants : 350
Traités en 353.67 sec : 653, restants : 300
Traités en 380.77 sec : 703, restants : 250
Traités en 407.12 sec : 753, restants : 200
Traités en 433.21 sec : 803, restants : 150
Traités en 459.06 sec : 853, restants : 100
Traités en 485.65 sec : 903, restants : 50
Traités en 511.58 sec : 953, restants : 0


,Uniprot,Gene description,Organ,GO
0,Q13103,Secreted phosphoprotein 2,Liver,GO:0062023;GO:0005788;GO:0005576;GO:0031089;GO...
1,P02765,Alpha 2-HS glycoprotein,Liver,GO:0072562;GO:0062023;GO:0005788;GO:0070062;GO...
2,P36980,Complement factor H related 2,Liver,GO:0005576;GO:0005615;GO:0032991;GO:0001851;GO...
3,P11226,Mannose binding lectin 2,Liver,GO:0009986;GO:0005581;GO:0009897;GO:0005576;GO...
4,P00740,Coagulation factor IX,Liver,GO:0062023;GO:0005788;GO:0070062;GO:0005576;GO...
...,...,...,...,...
973,Q9HAW9,UDP glucuronosyltransferase family 1 member A8,Liver,GO:0005783;GO:0005789;GO:0019899;GO:0004857;GO...
974,Q9BQB6,Vitamin K epoxide reductase complex subunit 1,Liver,GO:0005783;GO:0005789;GO:0048038;GO:0047058;GO...
975,Q96DN2,Von Willebrand factor C and EGF domains,Liver,GO:0005737;GO:0005576;GO:0005509;GO:0098586
976,Q3MJ13,WD repeat domain 72,Liver,GO:0005737;GO:0005768;GO:0005634;GO:0036305;GO...


In [ ]:
df_brain = pd.read_csv('/content/tissue_category_rna_testis_Tissue.tsv', sep='\t')
df_brain_reduced = df_brain[["Uniprot", "Gene description"]]
df_brain_reduced["Organ"] = ["Testis"] * len(df_brain_reduced)
df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]
df_brain_reduced = df_brain_reduced.dropna()
brain_go_dict = getGoTermsFromUniprot(df_brain_reduced["Uniprot"].tolist())
df_brain_reduced["GO"] = df_brain_reduced["Uniprot"].map(brain_go_dict)
df_brain_reduced.to_csv("Testis_go.csv", index=False)
df_brain_reduced

<ipython-input-79-1faa9e3071c8>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Organ"] = ["Testis"] * len(df_brain_reduced)
<ipython-input-79-1faa9e3071c8>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_brain_reduced["Uniprot"] = df_brain_reduced["Uniprot"].str.split(', ').str[0]


Traités en 22.39 sec : 45, restants : 1850
Traités en 47.22 sec : 95, restants : 1800
Traités en 72.16 sec : 145, restants : 1750
Traités en 96.94 sec : 195, restants : 1700
Traités en 121.98 sec : 245, restants : 1650
Traités en 146.98 sec : 295, restants : 1600
Traités en 171.80 sec : 345, restants : 1550
Traités en 196.65 sec : 395, restants : 1500
Traités en 221.41 sec : 445, restants : 1450
Traités en 246.38 sec : 495, restants : 1400
Traités en 271.50 sec : 545, restants : 1350
Traités en 296.54 sec : 595, restants : 1300
Traités en 321.30 sec : 645, restants : 1250
Traités en 346.15 sec : 695, restants : 1200
Traités en 370.99 sec : 745, restants : 1150
Traités en 395.92 sec : 795, restants : 1100
Traités en 421.02 sec : 845, restants : 1050
Traités en 446.28 sec : 895, restants : 1000
Traités en 472.07 sec : 945, restants : 950
Traités en 497.53 sec : 995, restants : 900
Traités en 523.82 sec : 1045, restants : 850
Traités en 548.87 sec : 1095, restants : 800
Traités en 574.28 

,Uniprot,Gene description,Organ,GO
0,Q8WW32,High mobility group box 4,Testis,GO:0005694;GO:0005634;GO:0008301;GO:0006357
1,P0DPH9,Chromosome X open reading frame 51B,Testis,
2,Q5T871,Late cornified envelope like proline rich 1,Testis,
3,P09430,Transition protein 1,Testis,GO:0001673;GO:0000786;GO:0005634;GO:0003677;GO...
4,Q96J77,TPD52 like 3,Testis,GO:0005737
...,...,...,...,...
1987,O75541,Zinc finger protein 821,Testis,GO:0005634;GO:0003700;GO:0046872;GO:0000978;GO...
1988,Q9UHR6,Zinc finger HIT-type containing 2,Testis,GO:0046872
1989,Q8NBB4,Zinc finger and SCAN domain containing 1,Testis,GO:0000785;GO:0005634;GO:0000981;GO:0046872;GO...
1990,Q9H900,Zwilch kinetochore protein,Testis,GO:0005829;GO:0000776;GO:0005828;GO:1990423;GO...


In [ ]:
df_liver = pd.read_csv('/content/Liver_go.csv', sep=',')
df_testis = pd.read_csv('/content/Testis_go.csv', sep=',')
df_kidney = pd.read_csv('/content/kidney_go.csv', sep=',')
df_heart = pd.read_csv('/content/heart_go.csv', sep=',')
df_brain = pd.read_csv('/content/brain_go.csv', sep=',')
df_all = pd.concat([df_liver, df_testis, df_kidney, df_heart, df_brain])
df_all.to_csv("all_go.csv", index=False)
df_all

,Uniprot,Gene description,Organ,GO
0,Q13103,Secreted phosphoprotein 2,Liver,GO:0062023;GO:0005788;GO:0005576;GO:0031089;GO...
1,P02765,Alpha 2-HS glycoprotein,Liver,GO:0072562;GO:0062023;GO:0005788;GO:0070062;GO...
2,P36980,Complement factor H related 2,Liver,GO:0005576;GO:0005615;GO:0032991;GO:0001851;GO...
3,P11226,Mannose binding lectin 2,Liver,GO:0009986;GO:0005581;GO:0009897;GO:0005576;GO...
4,P00740,Coagulation factor IX,Liver,GO:0062023;GO:0005788;GO:0070062;GO:0005576;GO...
...,...,...,...,...
2148,Q96CK0,Zinc finger protein 653,Brain,GO:0005576;GO:0005634;GO:0050682;GO:0003677;GO...
2149,Q7Z570,Zinc finger protein 804A,Brain,GO:0005737;GO:1901588;GO:0043198;GO:0043197;GO...
2150,O75541,Zinc finger protein 821,Brain,GO:0005634;GO:0003700;GO:0046872;GO:0000978;GO...
2151,Q8NBB4,Zinc finger and SCAN domain containing 1,Brain,GO:0000785;GO:0005634;GO:0000981;GO:0046872;GO...


In [ ]:
# Shuffle le DataFrame
shuffled_df = df_all.sample(frac=1, random_state=42)  # random_state pour la reproductibilité

# Divisez le DataFrame en deux ensembles : 90% et 10%
split_index = int(0.9 * len(shuffled_df))
df_train_all = shuffled_df.iloc[:split_index]
sampling_df = shuffled_df.iloc[split_index:]

sampling_df.to_csv("sampling_go.csv", index=False)


In [ ]:
split_index2 = int(0.8 * len(df_train_all))
df_train = df_train_all.iloc[:split_index2]
df_test = df_train_all.iloc[split_index2:]

df_train.to_csv("train_all.csv", index=False)
df_test.to_csv("test_all.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import random

# Supposons que vous avez un DataFrame nommé df
# df = pd.read_csv('votre_fichier.csv')  # Remplacez par votre source de données

# Shuffle le DataFrame
shuffled_df = sampling_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Liste des organes uniques
organes = shuffled_df['Organ'].unique()

# Fonction pour créer un sous-DataFrame avec une majorité d'un organe
def create_majority_subset(df, organe, majority_fraction, max_sequences=50):
    # Filtrer les lignes de l'organe spécifié
    organe_df = df[df['Organ'] == organe]
    other_df = df[df['Organ'] != organe]

    # Calculer le nombre de lignes pour la majorité
    n_majority = int(min(len(organe_df) * majority_fraction, max_sequences * 0.9))
    n_other = min(max_sequences - n_majority, len(other_df))

    # Sélectionner aléatoirement les lignes
    majority_subset = organe_df.sample(n=n_majority, random_state=42)
    other_subset = other_df.sample(n=n_other, random_state=42)

    # Combiner les sous-ensembles
    combined_subset = pd.concat([majority_subset, other_subset]).sample(frac=1, random_state=42).reset_index(drop=True)
    return combined_subset

# Créer et sauvegarder les sous-DataFrames
for organe in organes:
    # Générer une fraction aléatoire entre 0.4 et 0.6
    majority_fraction = random.uniform(0.4, 0.6)
    subset_df = create_majority_subset(shuffled_df, organe, majority_fraction)

    # Sauvegarder le sous-DataFrame en CSV
    subset_df.to_csv(f'majorité_{organe}.csv', index=False)
    print(f"Sous-DataFrame pour {organe} sauvegardé avec {majority_fraction * 100:.2f}% de majorité.")


Sous-DataFrame pour Brain sauvegardé avec 42.70% de majorité.
Sous-DataFrame pour Heart sauvegardé avec 42.85% de majorité.
Sous-DataFrame pour Kidney sauvegardé avec 57.35% de majorité.
Sous-DataFrame pour Testis sauvegardé avec 46.48% de majorité.
Sous-DataFrame pour Liver sauvegardé avec 44.31% de majorité.


In [ ]:
# Fonction pour récupérer la séquence protéique à partir d'un identifiant UniProt
def get_protein_sequence(uniprot_id):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(url)
    if response.status_code == 200:
        if '\n' in response.text:
            # Extraire la séquence du contenu de la réponse
            sequence = response.text.split('\n', 1)[1].replace('\n', '')
            return sequence
        else:
            # Gérer le cas où il n'y a pas de saut de ligne
            print(f"Unexpected response format for {uniprot_id}: {response.text}")
            return None
    else:
        print(f"Error fetching data for {uniprot_id}: {response.status_code}")
        return None

def introduce_random_mutations(sequence, num_mutations=5):
    if sequence is None or sequence == "" or sequence == "NaN":
        return sequence
    mutated_sequence = list(sequence)
    # Store the original length for accurate indexing
    original_length = len(sequence)

    for _ in range(num_mutations):
        mutation_type = random.choice(['substitution', 'deletion', 'insertion'])
        # Ensure the index is within the bounds of the original length
        position = random.randint(1, original_length - 2)

        if mutation_type == 'substitution':
            mutated_sequence[position] = random.choice('ACDEFGHIKLMNPQRSTVWY')
        elif mutation_type == 'deletion':
            # Adjust position to delete if it is too large
            if position >= len(mutated_sequence):
                position = len(mutated_sequence) - 1

            mutated_sequence.pop(position)
        elif mutation_type == 'insertion':
            mutated_sequence.insert(position, random.choice('ACDEFGHIKLMNPQRSTVWY'))

    return ''.join(mutated_sequence)

In [ ]:
import pandas as pd
import numpy as np
import random
import requests

df = sampling_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Supposons que vous avez un DataFrame nommé df
# df = pd.read_csv('votre_fichier.csv')  # Remplacez par votre source de données

# Ajouter une colonne pour les séquences protéiques
df['ProteinSequence'] = df['Uniprot'].apply(get_protein_sequence)




Unexpected response format for Q08AF8: 


In [ ]:
# Introduire des mutations aléatoires dans les séquences protéiques
df['MutatedSequence'] = df['ProteinSequence'].apply(introduce_random_mutations)

In [ ]:
# Shuffler le DataFrame
shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Liste des organes uniques
organes = shuffled_df['Organ'].unique()

# Fonction pour créer un sous-DataFrame avec une majorité d'un organe
def create_majority_subset(df, organe, majority_fraction, max_sequences=50):
    # Filtrer les lignes de l'organe spécifié
    organe_df = df[df['Organ'] == organe]
    other_df = df[df['Organ'] != organe]

    # Calculer le nombre de lignes pour la majorité
    n_majority = int(min(len(organe_df) * majority_fraction, max_sequences * 0.9))
    n_other = min(max_sequences - n_majority, len(other_df))

    # Sélectionner aléatoirement les lignes
    majority_subset = organe_df.sample(n=n_majority, random_state=42)
    other_subset = other_df.sample(n=n_other, random_state=42)

    # Combiner les sous-ensembles
    combined_subset = pd.concat([majority_subset, other_subset]).sample(frac=1, random_state=42).reset_index(drop=True)

    # Garder uniquement la colonne 'MutatedSequence'
    combined_subset = combined_subset[['MutatedSequence']]

    return combined_subset

# Créer et sauvegarder les sous-DataFrames
for organe in organes:
    # Générer une fraction aléatoire entre 0.4 et 0.6
    majority_fraction = random.uniform(0.4, 0.6)
    subset_df = create_majority_subset(shuffled_df, organe, majority_fraction)

    # Sauvegarder le sous-DataFrame en CSV
    subset_df.to_csv(f'majorité_{organe}.csv', index=False)
    print(f"Sous-DataFrame pour {organe} sauvegardé avec {majority_fraction * 100:.2f}% de majorité.")

Sous-DataFrame pour Liver sauvegardé avec 42.32% de majorité.
Sous-DataFrame pour Brain sauvegardé avec 45.86% de majorité.
Sous-DataFrame pour Testis sauvegardé avec 42.32% de majorité.
Sous-DataFrame pour Kidney sauvegardé avec 46.02% de majorité.
Sous-DataFrame pour Heart sauvegardé avec 43.55% de majorité.


In [ ]:
def giveRandomNameForSequence():
  import random
  import string
  letters = string.ascii_lowercase
  return ''.join(random.choice(letters) for i in range(10))
  return name

def giveRandomPatientId():
  import random
  import string
  digits = string.digits
  return ''.join(random.choice(digits) for i in range(5))
  return name

def createMultiFastaFileFromDf(df, filename="output.fasta", sequence_col="MutatedSequence", id_col="Uniprot"):
    """
    Creates a multi-FASTA file from a Pandas DataFrame.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    filename (str): The name of the output FASTA file.
    sequence_col (str): The name of the column containing protein sequences.
    id_col (str): The name of the column containing protein IDs (UniProt IDs in this case).
    """

    with open(filename, "w") as fasta_file:
        patient_id = giveRandomPatientId()
        for index, row in df.iterrows():
            # Get sequence and ID
            sequence = row[sequence_col]
            identifier = "patient_" + patient_id + "_Protein_" + str(index) + "_" + giveRandomNameForSequence()

            # Write in FASTA format
            fasta_file.write(f">{identifier}\n")
            fasta_file.write(f"{sequence}\n")

    print(f"Multi-FASTA file '{filename}' created successfully.")


In [ ]:
df = pd.read_csv('/content/majorité_Heart.csv', sep=',')
createMultiFastaFileFromDf(df, filename="heart.fasta", sequence_col="MutatedSequence", id_col="Uniprot")

df = pd.read_csv('/content/majorité_Liver.csv', sep=',')
createMultiFastaFileFromDf(df, filename="liver.fasta", sequence_col="MutatedSequence", id_col="Uniprot")

df = pd.read_csv('/content/majorité_Testis.csv', sep=',')
createMultiFastaFileFromDf(df, filename="testis.fasta", sequence_col="MutatedSequence", id_col="Uniprot")

df = pd.read_csv('/content/majorité_Kidney.csv', sep=',')
createMultiFastaFileFromDf(df, filename="kidney.fasta", sequence_col="MutatedSequence", id_col="Uniprot")

df = pd.read_csv('/content/majorité_Brain.csv', sep=',')
createMultiFastaFileFromDf(df, filename="brain.fasta", sequence_col="MutatedSequence", id_col="Uniprot")

Multi-FASTA file 'heart.fasta' created successfully.
Multi-FASTA file 'liver.fasta' created successfully.
Multi-FASTA file 'testis.fasta' created successfully.
Multi-FASTA file 'kidney.fasta' created successfully.
Multi-FASTA file 'brain.fasta' created successfully.
